# Leave-one-out figures + benchmark table

Visualization and benchmark table for the leave-one-out experiment, from the saved predictions
under `results/realdata/` (per-method folders + the per-baseline `metrics_*.json` +
`benchmark_detailed.csv`); the ground truth is reloaded from `data/`. One 2x3 figure per
dataset x dimension.

TIGON resolves strictly inside `tigon_final/` (`_loo_paths.STRICT_METHODS`), so it can never be
filled in from the exploratory calibration predictions that share its filenames; a method with no
file keeps its panel, marked `run pending`.

In [ ]:
import os, sys, csv, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


def _bootstrap():
    here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
    # this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
    for d in (here, os.path.abspath(os.path.join(here, os.pardir, os.pardir, "tools"))):
        if d not in sys.path:
            sys.path.insert(0, d)
    import _repo
    _repo.add_paths()
    return _repo


P = _bootstrap()
REPO = P.REPO
from uotreg import datasets as _data

_here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
# this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
for _d in (_here, os.path.abspath(os.path.join(_here, os.pardir, os.pardir, "tools"))):
    if _d not in sys.path:
        sys.path.insert(0, _d)
import _loo_paths as _P            # per-method folders + legacy flat fallback
from uotreg.metrics import w2, emd, mmd_rbf
# the shipped LOO artifacts; switch to _P.results_root() to prefer your own `new_results/` run
INDIR = _P.results_root(shipped=True)

# ---- TEXT SIZE -------------------------------------------------------------------------------
# Same scheme as the outlier / divergence figures: the figure goes into the draft at
# `width=\linewidth` (= 6.5in), so LaTeX rescales it by 6.5 / figsize-width and what the reader sees
# is  printed pt = fontsize x 6.5 / figsize-width  (0.54x for the 12in-wide 2x3 below). FS multiplies
# every font size at once; the per-element sizes stay written next to what they control.
FS = globals().get("FS", 1.6)
plt.rcParams.update({"font.size": 11 * FS, "axes.titlesize": 11 * FS, "axes.labelsize": 11 * FS,
                     "figure.titlesize": 13 * FS, "xtick.labelsize": 10 * FS,
                     "ytick.labelsize": 10 * FS, "legend.fontsize": 10 * FS})

PRED_C, TRUE_C = "tab:purple", "0.72"      # predicted cloud / held-out ground truth
N_SHOW = 1500                              # cells drawn per panel (truth and prediction each)

# What an empty panel says. TIGON resolves strictly inside `tigon_final/` (see `_loo_paths`), so
# while its production run is on the cluster its panels are deliberately blank rather than filled
# from the exploratory calibration files that share the same names in the flat directory.
PENDING_NOTE = "run pending"

# ---- axis ranges --------------------------------------------------------------------------------
# Per (dataset, dim) axis box as ((x0, x1), (y0, y1)). The values below are the TRUE AUTOSCALE ranges
# -- i.e. exactly what matplotlib picks from truth + every prediction -- so out of the box the panels
# look untouched. Edit any entry to crop/expand that combination; set an entry to None (or delete it)
# to fall back to autoscale. `fig_loo(..., lim=((x0,x1),(y0,y1)))` still overrides per call.
#
# Note the ranges are driven by the widest cloud, which for embryoid is MMFM's far tail -- that is
# why embryoid d=20/50 reach down to y ~ -17 while the truth only spans y ~ [-5, 10].
LIMS = {
    ("embryoid",  10): ((-11.6, 17.9), (-7.4, 10.6)),
    ("embryoid",  20): ((-11.8, 19.0), (-8.2, 11.5)),
    ("embryoid",  50): ((-10.8, 19.5), (-8.2, 11.5)),
    ("statefate", 10): ((-10.6, 22.2), (-9.2, 25.4)),
    ("statefate", 20): ((-10.4, 21.1), (-8.0, 20.0)),
    ("statefate", 50): ((-12.1, 24.0), (-7.7, 20.0)),
}

# ---- read-out direction ------------------------------------------------------------------------
TIGON_DIR = "bwd"   # "bwd" = the authors' canonical read-out | "fwd"
MMFM_DIR  = "bwd"    # "bwd" | "fwd"  -- see the caveat below
#
# CAVEAT: `loo_{embryoid,statefate}.py` save with `pred_..._{nm.split()[0]}.npy`, so "MMFM fwd" and
# "MMFM bwd" BOTH write `pred_..._MMFM.npy` and the bwd (written second) wins. Only one MMFM file is
# therefore on disk, and it is the BACKWARD read-out. `MMFM_DIR="fwd"` looks for
# `pred_..._MMFMfwd.npy` and, not finding it, falls back to the bwd file with a printed warning.
# To get both, change that save line to key on the full name, e.g.
#     np.save(os.path.join(RESULTS, f"pred_{DATASET}{DIM}_Day{HELD}_{nm.replace(' ', '')}.npy"), ...)
# and re-run the LOO (or just the MMFM read-out) -- TIGON already writes `TIGON` + `TIGONfwd`.

# panel order: (label, base method key). The direction switches are applied in `_pred_key`.
METHODS = [("ours (UOTReg)", "UOTReg"), ("OT", "OT"), ("naive midpoint", "Naive1"),
           ("MMFM", "MMFM"), ("MioFlow", "MioFlow"), ("TIGON", "TIGON")]

## Loaders
`_pred` reads one prediction (applying the fwd/bwd switch and falling back where a variant is
missing); `_w2` looks the W2 up in `benchmark_detailed.csv`, then in the per-method
`metrics_*.json` (which is where the fwd variants live); `_truth` reloads the held-out snapshot.

In [ ]:
def _pred_key(key):
    """Apply the direction switches to a base method key."""
    if key == "TIGON" and TIGON_DIR == "fwd":
        return "TIGONfwd"
    if key == "MMFM" and MMFM_DIR == "fwd":
        return "MMFMfwd"
    return key


def _pred(dataset, dim, held, key):
    """The saved prediction cloud, or None if absent. Falls back when a fwd variant is missing."""
    want = _pred_key(key)
    p = _P.pred_path(INDIR, dataset, dim, held, want)
    if not os.path.exists(p) and want != key:
        alt = _P.pred_path(INDIR, dataset, dim, held, key)
        if os.path.exists(alt):
            print(f"   ! {want} not on disk for {dataset}{dim} Day{held} -- using {key} "
                  f"(the saved MMFM file is the BACKWARD read-out; see the CAVEAT above)")
            p, want = alt, key
    if not os.path.exists(p):
        print(f"   ! missing {os.path.basename(p)}"); return None, want
    return np.load(p), want


_CSV = None
CSV_PATH = os.path.join(INDIR, "benchmark_detailed.csv")

# Filled by `metrics()` in the table section below. Declared HERE because `_w2` prefers it when it is
# populated -- so a figure redrawn after the table carries the table's own numbers.
METRICS_ROWS, METRICS_SRC = {}, {}


def _csv_rows():
    global _CSV
    if _CSV is None:
        _CSV = (list(csv.DictReader(open(CSV_PATH))) if os.path.exists(CSV_PATH) else [])
    return _CSV


def _w2(dataset, dim, held, key, label=None):
    """W2 mean for a panel title, from the first source that has it: this session's scoring (the
    metrics section below), then `benchmark_detailed.csv`, then the method's own metrics json.

    The csv does not cover every cell -- TIGON was scored after it was written, so its rows come
    from its own json. The table section below rescores everything uniformly; these panel numbers
    are for the figure alone."""
    if label is not None:
        cell = METRICS_ROWS.get((dataset, int(dim), float(held)))
        if cell and label in cell:
            return cell[label]["W2"][0]
    for r in _csv_rows():
        if (r["dataset"] == dataset and int(r["dim"]) == int(dim)
                and float(r["held"]) == float(held) and r["method"] == key):
            return float(r["W2_mean"])
    j = _P.metrics_path(INDIR, dataset, dim, held, key)
    if os.path.exists(j):
        return float(json.load(open(j))["metrics"]["W2"]["mean"])
    return None


def _truth(dataset, dim, held):
    """The true held-out cells (not saved by the LOO files -- reloaded from the dataset).
    Returns None when the data cannot be loaded -- statefate at d=50 needs the large .h5ad that
    does not ship with the repository (see the README); that cell is skipped, not an error."""
    try:
        ds = getattr(_data, f"load_{dataset}")(P.DATA_DIR, d=dim)
    except (FileNotFoundError, ImportError) as e:
        print(f"   ! {dataset} d={dim}: ground truth unavailable ({e}) -- skipping")
        return None
    tp = ds.timepoints.tolist()
    return np.asarray(ds.arrays[tp.index(float(held))], np.float32)

## The figure
2x3, one panel per method, grey truth under each purple prediction, W2 in the panel title, and a
purple/grey key between the suptitle and the panels.

In [ ]:
def fig_loo(dataset, dim, held, methods=None, truth=None, n_show=N_SHOW, lim=None):
    """One LOO panel figure for a (dataset, dim, held-day) combination.

    Axis ranges come from `LIMS[(dataset, dim)]`; `lim=((x0,x1),(y0,y1))` overrides it, and if
    neither is set the axes autoscale."""
    methods = methods or METHODS
    rng = np.random.default_rng(0)
    sub = lambda X: (X if len(X) <= n_show else X[rng.choice(len(X), n_show, replace=False)])
    T = sub(np.asarray(truth if truth is not None else _truth(dataset, dim, held)))

    fig, axes = plt.subplots(2, 3, figsize=(12, 8.2), dpi=140, sharex=True, sharey=True)
    for ax, (label, key) in zip(axes.ravel(), methods):
        g, used = _pred(dataset, dim, held, key)
        ax.scatter(T[:, 0], T[:, 1], s=6, c=TRUE_C, alpha=0.45)
        if g is None:
            # No prediction on disk (TIGON, until the cluster run is pulled back into tigon_final/).
            # The panel keeps its place in the grid and shows the truth, with the slot NAMED as
            # pending -- an unlabelled grey cloud would otherwise read as this method's prediction.
            ax.set_title(f"{label}  ({PENDING_NOTE})", fontsize=11 * FS)
            ax.text(0.5, 0.5, PENDING_NOTE, transform=ax.transAxes, ha="center", va="center",
                    fontsize=11 * FS, color="0.35",
                    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="0.7", alpha=0.85))
            continue
        P = sub(np.asarray(g))
        ax.scatter(P[:, 0], P[:, 1], s=6, c=PRED_C, alpha=0.55)
        w = _w2(dataset, dim, held, used, label=label)
        ax.set_title(f"{label}" + (f"  (W2 {w:.2f})" if w is not None else ""), fontsize=11 * FS)
    for ax in axes[:, 0]:
        ax.set_ylabel("PC2")
    for ax in axes[-1, :]:
        ax.set_xlabel("PC1")
    box = lim if lim is not None else LIMS.get((dataset, int(dim)))
    if box is not None:                                  # shared axes -> setting one sets all
        axes[0, 0].set_xlim(*box[0]); axes[0, 0].set_ylim(*box[1])

    # header stack: suptitle -> legend row -> panel titles -> panels
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.suptitle(f"{dataset} d={dim}, held Day {held}: predicted vs true held-out cells",
                 fontsize=13 * FS, y=0.985)
    fig.legend(handles=[
        Line2D([0], [0], marker="o", ls="", mfc=PRED_C, mec="none", label="predicted"),
        Line2D([0], [0], marker="o", ls="", mfc=TRUE_C, mec="none", label="ground truth (held-out)")],
        loc="upper center", ncol=2, frameon=False, fontsize=10 * FS, bbox_to_anchor=(0.5, 0.945))
    plt.show()
    return fig

## Draw all six combinations
embryoid Day 13.5 and statefate Day 4.0, at d = 10 / 20 / 50. The dataset is loaded once per
(dataset, dim) so the ground truth is not re-read per figure.

In [ ]:
COMBOS = [("embryoid", "13.5"), ("statefate", "4.0")]
DIMS = [10, 20, 50]

for dataset, held in COMBOS:
    for dim in DIMS:
        print(f"\n[{dataset} d={dim} Day{held}]  TIGON={TIGON_DIR}  MMFM={MMFM_DIR}")
        T = _truth(dataset, dim, held)
        if T is None:
            continue
        fig_loo(dataset, dim, held, truth=T)

## Metrics table
The same predictions the figures draw, re-scored with ONE seeded scorer (MMD / EMD / W2 over
`BENCH_REPEATS` equal-size draws) so every row is comparable -- the per-method `metrics_*.json`
were written by different scripts with different draw counts. Covers the full grid (embryoid
7.5 / 13.5 / 19.5, statefate 4.0, at d = 10 / 20 / 50), not just the days the figures draw.

In [ ]:
BENCH_REPEATS = 30
BENCH_N = 1000
SEED = 0
ALL_HELD = {"embryoid": [7.5, 13.5, 19.5], "statefate": [4.0]}
METRIC_FNS = {"MMD": mmd_rbf, "EMD": emd, "W2": w2}


def _score(pred, truth, seed=SEED):
    """MMD/EMD/W2 over equal-size bootstrap draws. Seeded -> the table is reproducible."""
    rng = np.random.default_rng(seed)
    pred, truth = np.asarray(pred), np.asarray(truth)
    acc = {k: [] for k in METRIC_FNS}
    for _ in range(BENCH_REPEATS):
        ref = truth[rng.integers(0, len(truth), BENCH_N)]
        p = pred[rng.integers(0, len(pred), BENCH_N)]
        for k, fn in METRIC_FNS.items():
            acc[k].append(fn(p, ref))
    return {k: (float(np.mean(v)), float(np.std(v))) for k, v in acc.items()}


def metrics(datasets=None, dims=None, held=None, methods=None, verbose=True):
    """Score every (dataset, dim, held day, method) whose prediction is on disk.

    Returns {(ds, dim, held): {label: {metric: (mean, std)}}}. Missing cells are skipped, so this
    runs while TIGON is still on the cluster."""
    datasets = datasets or [d for d, _ in COMBOS]
    dims = dims or DIMS
    held = held or ALL_HELD
    methods = methods or METHODS
    rows, prov = {}, {}
    for ds in datasets:
        for dim in dims:
            for h in held[ds]:
                truth = _truth(ds, dim, h)
                if truth is None:
                    continue
                cell, src_ = {}, {}
                for lab, key in methods:
                    g, want = _pred(ds, dim, h, key)
                    if g is None:
                        continue
                    cell[lab] = _score(g, truth)
                    p = _P.pred_path(INDIR, ds, dim, h, want)
                    src_[lab] = ("FLAT" if os.path.dirname(p) == INDIR
                                 else os.path.basename(os.path.dirname(p)))
                if cell:
                    rows[(ds, dim, h)] = cell
                    prov[(ds, dim, h)] = src_
                    if verbose:
                        flat = [m for m, s in src_.items() if s == "FLAT"]
                        print(f"[{ds} d={dim} Day{h}] scored {len(cell)}"
                              + (f"   ! read from the FLAT dir: {flat}" if flat else ""))
    METRICS_ROWS.update(rows); METRICS_SRC.update(prov)
    return rows


ROWS = metrics()

## The table

In [ ]:
def table(rows=None, metric_order=("MMD", "EMD", "W2")):
    rows = rows if rows is not None else METRICS_ROWS
    for (ds, dim, h), cell in sorted(rows.items()):
        print(f"\n[{ds} d={dim} Day{h}]   lower = better")
        print("   " + f"{'method':16s}" + "".join(f"{k:>10}" for k in metric_order) + "   source")
        best = {k: min(v[k][0] for v in cell.values()) for k in metric_order}
        for lab, _ in METHODS:
            if lab not in cell:
                # Kept as a visible blank row: a method silently dropped from the table looks like a
                # method that was never run. TIGON sits here until `tigon_final/` is populated.
                print("   " + f"{lab:16s}" + "".join(f"{'--':>10}" for _ in metric_order)
                      + f"   {PENDING_NOTE}")
                continue
            cells = "".join(f"{cell[lab][k][0]:>9.3f}" + ("*" if cell[lab][k][0] == best[k] else " ")
                            for k in metric_order)
            print("   " + f"{lab:16s}" + cells + f"   {METRICS_SRC[(ds, dim, h)][lab]}")
    print("\n  * = best in column (among the methods present).  source = which folder the prediction"
          "\n  came from; FLAT = not yet migrated, '" + PENDING_NOTE + "' = nothing on disk for that "
          "method yet.")


table()